# Stage 5 — Scoring Output

Scores the full 50,000-member population, applies isotonic calibration,
assigns risk tiers and recommended modality, and produces the outreach list.

## Step 1 — Load model, calibrator and features

In [1]:
import pandas as pd
import numpy as np
import pickle, json, os
from pathlib import Path

data_dir    = Path().resolve().parent / 'data'
outputs_dir = Path().resolve().parent / 'outputs'
os.makedirs(outputs_dir, exist_ok=True)

# Load model
with open(outputs_dir / 'model.pkl', 'rb') as f:
    model = pickle.load(f)

# Load calibrator
with open(outputs_dir / 'calibrator.pkl', 'rb') as f:
    calibrator = pickle.load(f)

# Reference metrics from Stage 4
with open(outputs_dir / 'eval_metrics.json') as f:
    eval_metrics = json.load(f)

print('Model loaded     best iteration: {}'.format(model.best_iteration))
print('Stage 4 metrics  ROC-AUC: {}   PR-AUC: {}   Recall@top20%: {}'.format(
    eval_metrics['roc_auc'], eval_metrics['pr_auc'], eval_metrics['recall_top20pct']))

# Load features — must include interaction features from Stage 2
features = pd.read_parquet(data_dir / 'features' / 'features.parquet')
features['condition_cluster'] = features['condition_cluster'].astype(str)
print('Feature matrix: {}'.format(features.shape))

FEATURE_COLS = [
    'age', 'age_band', 'gender', 'plan_type', 'employer_group_size', 'tenure_months',
    'has_msk_flag', 'has_metabolic_flag', 'has_mh_flag',
    'comorbidity_count', 'condition_cluster',
    'total_claims_6m', 'total_spend_6m',
    'gp_visits_6m', 'specialist_visits_6m', 'allied_health_claims_6m',
    'days_since_last_allied', 'allied_health_utilisation_rate',
    'specialist_to_gp_ratio', 'zero_allied_health_flag', 'high_gp_low_allied',
    'sessions_remaining_physio', 'sessions_remaining_chiro',
    'sessions_remaining_dietetics', 'sessions_remaining_psychology',
    'any_benefits_remaining', 'benefit_utilisation_rate',
    # Interaction features (Lever 2) — must match Stage 4 exactly
    'msk_zero_allied', 'metabolic_zero_allied', 'mh_zero_allied',
    'comorbid_zero_allied',
    'age_zero_allied', 'bronze_high_comorbid',
]
CAT_COLS = ['age_band', 'gender', 'plan_type', 'employer_group_size', 'condition_cluster']

assert len(FEATURE_COLS) == model.num_feature(), \
    'FEATURE_COLS count ({}) does not match model feature count ({}) — check Stage 2 and 4'.format(
        len(FEATURE_COLS), model.num_feature())

X_all = features[FEATURE_COLS].copy()
for col in CAT_COLS:
    X_all[col] = X_all[col].astype('category')

print('X_all shape: {}  ({} features)'.format(X_all.shape, len(FEATURE_COLS)))

Model loaded     best iteration: 362
Stage 4 metrics  ROC-AUC: 0.7417   PR-AUC: 0.372   Recall@top20%: 0.4458
Feature matrix: (50000, 34)
X_all shape: (50000, 33)  (33 features)


## Step 2 — Score full population

In [2]:
# Raw LightGBM scores
raw_scores = model.predict(X_all, num_iteration=model.best_iteration)

# Calibrated scores — apply isotonic calibrator fitted on validation set in Stage 4
calibrated_scores = calibrator.predict(raw_scores)

print('Score distribution (raw):')
print('  min={:.4f}  p25={:.4f}  median={:.4f}  p75={:.4f}  max={:.4f}'.format(
    raw_scores.min(),
    np.percentile(raw_scores, 25),
    np.percentile(raw_scores, 50),
    np.percentile(raw_scores, 75),
    raw_scores.max()
))
print('Score distribution (calibrated):')
print('  min={:.4f}  p25={:.4f}  median={:.4f}  p75={:.4f}  max={:.4f}'.format(
    calibrated_scores.min(),
    np.percentile(calibrated_scores, 25),
    np.percentile(calibrated_scores, 50),
    np.percentile(calibrated_scores, 75),
    calibrated_scores.max()
))

Score distribution (raw):
  min=0.0372  p25=0.2887  median=0.4199  p75=0.5777  max=0.8660
Score distribution (calibrated):
  min=0.0000  p25=0.0980  median=0.1318  p75=0.2383  max=1.0000


## Step 3 — Assign risk tiers

Thresholds set at the 80th and 90th percentiles of the **raw score distribution**.
Raw scores are continuous with no isotonic plateau ties, giving clean 10%/10%/80% splits.
Calibrated scores are stored separately as probability estimates for reporting.

In [3]:
p80 = np.percentile(raw_scores, 80)
p90 = np.percentile(raw_scores, 90)

def assign_tier(score):
    if score >= p90:  return 'High'
    if score >= p80:  return 'Medium'
    return 'Low'

risk_tiers = np.vectorize(assign_tier)(raw_scores)

tier_counts = pd.Series(risk_tiers).value_counts()
print('Risk tier distribution:')
for tier in ['High', 'Medium', 'Low']:
    n = tier_counts.get(tier, 0)
    print('  {:<8}  {:>6,}  ({:.1f}%)'.format(tier, n, n/len(risk_tiers)*100))
print('  Threshold High   >= {:.4f}'.format(p90))
print('  Threshold Medium >= {:.4f}'.format(p80))

Risk tier distribution:
  High       5,000  (10.0%)
  Medium     5,000  (10.0%)
  Low       40,000  (80.0%)
  Threshold High   >= 0.7276
  Threshold Medium >= 0.6534


## Step 4 — Recommend modality

Primary driver is condition cluster — clinically appropriate first-line service:
MSK → Physiotherapy, Metabolic → Dietetics, MH → Psychology.
Mixed and Healthy fall back to the benefit type with most sessions remaining
(clinical priority tiebreak: dietetics > psychology > physiotherapy > chiropractic).
If the cluster-preferred modality is exhausted, falls back to largest remaining gap.

In [4]:
MODALITY_PRIORITY = ['sessions_remaining_dietetics', 'sessions_remaining_psychology',
                     'sessions_remaining_physio', 'sessions_remaining_chiro']
MODALITY_LABELS   = {'sessions_remaining_psychology': 'Psychology',
                     'sessions_remaining_physio':      'Physiotherapy',
                     'sessions_remaining_dietetics':   'Dietetics',
                     'sessions_remaining_chiro':       'Chiropractic'}

# Condition-cluster preferred modality column
CLUSTER_PREFERRED = {
    'MSK':       'sessions_remaining_physio',
    'Metabolic': 'sessions_remaining_dietetics',
    'MH':        'sessions_remaining_psychology',
    'Mixed':     None,
    'Healthy':   None,
}

def recommend_modality(row):
    preferred_col = CLUSTER_PREFERRED.get(str(row['condition_cluster']))

    # Try cluster-preferred modality first
    if preferred_col and row[preferred_col] > 0:
        return MODALITY_LABELS[preferred_col]

    # Fallback: largest remaining sessions with clinical priority tiebreak
    remaining = {col: row[col] for col in MODALITY_PRIORITY}
    max_sessions = max(remaining.values())
    if max_sessions == 0:
        return 'None (exhausted)'
    for col in MODALITY_PRIORITY:
        if remaining[col] == max_sessions:
            return MODALITY_LABELS[col]

features['recommended_modality'] = features[MODALITY_PRIORITY + ['condition_cluster']].apply(
    recommend_modality, axis=1)

print('Recommended modality distribution:')
print(features['recommended_modality'].value_counts().to_string())

Recommended modality distribution:
recommended_modality
Dietetics           20909
Psychology          16285
Physiotherapy       12298
Chiropractic          501
None (exhausted)        7


## Step 5 — Flag benefit gap and plan design paradox

In [5]:
# sessions_remaining_total computed in Stage 2 via benefits pivot
sessions_remaining_total = features[MODALITY_PRIORITY].sum(axis=1)

# benefit_gap_flag: member has unused sessions (entitlement remains)
benefit_gap_flag = (sessions_remaining_total > 0).astype(int)

# nudge_signal: eligible for AND actively missing out on allied health
# Requires both unused sessions AND zero allied health claims in the past 6m.
# benefit_gap_flag alone fires for ~100% of members (most have unused entitlement).
# Adding zero_allied_health_flag narrows to the sub-population with the most
# to gain from outreach — those with available sessions they're not using at all.
nudge_signal = (
    (sessions_remaining_total > 0) & (features['zero_allied_health_flag'] == 1)
).astype(int)

# plan_design_flag: High-risk member with ZERO sessions remaining
# → entitlement was exhausted before need was met — plan design failure
plan_design_flag = (
    (pd.Series(risk_tiers) == 'High') & (sessions_remaining_total == 0)
).astype(int).values

_sep = '─' * 50
print(_sep)
print('  FLAG SUMMARY')
print(_sep)
print('  {:<30}  {:>8,}  ({:.1f}%)'.format(
    'benefit_gap_flag = 1', benefit_gap_flag.sum(), benefit_gap_flag.mean()*100))
print('  {:<30}  {:>8,}  ({:.1f}%)'.format(
    'nudge_signal = 1', nudge_signal.sum(), nudge_signal.mean()*100))
print('  {:<30}  {:>8,}  ({:.1f}%)'.format(
    'plan_design_flag = 1', plan_design_flag.sum(), plan_design_flag.mean()*100))
print(_sep)

──────────────────────────────────────────────────
  FLAG SUMMARY
──────────────────────────────────────────────────
  benefit_gap_flag = 1              49,993  (100.0%)
  nudge_signal = 1                   4,758  (9.5%)
  plan_design_flag = 1                   0  (0.0%)
──────────────────────────────────────────────────


## Step 6 — Assemble and save scored_members.csv

In [6]:
IDENTITY_COLS = ['member_id', 'age', 'gender', 'plan_type', 'employer_group_id',
                 'state', 'condition_cluster', 'comorbidity_count',
                 'has_msk_flag', 'has_metabolic_flag', 'has_mh_flag']

members_raw = pd.read_csv(data_dir / 'raw' / 'members.csv')

scored = members_raw[IDENTITY_COLS].copy()
scored['gp_visits_6m']              = features['gp_visits_6m'].values
scored['allied_health_claims_6m']   = features['allied_health_claims_6m'].values
scored['allied_health_utilisation_rate'] = features['allied_health_utilisation_rate'].values
scored['days_since_last_allied']    = features['days_since_last_allied'].values
scored['zero_allied_health_flag']   = features['zero_allied_health_flag'].values
scored['sessions_remaining_physio'] = features['sessions_remaining_physio'].values
scored['sessions_remaining_chiro']  = features['sessions_remaining_chiro'].values
scored['sessions_remaining_dietetics']  = features['sessions_remaining_dietetics'].values
scored['sessions_remaining_psychology'] = features['sessions_remaining_psychology'].values
scored['sessions_remaining_total']  = sessions_remaining_total.values
scored['recommended_modality']      = features['recommended_modality'].values
scored['risk_score']                = raw_scores          # raw score — primary ranking column
scored['risk_score_calibrated']     = calibrated_scores   # calibrated probability estimate
scored['risk_tier']                 = risk_tiers
scored['benefit_gap_flag']          = benefit_gap_flag.values
scored['nudge_signal']              = nudge_signal.values
scored['plan_design_flag']          = plan_design_flag

# Sort by raw risk score descending — highest risk first, no plateau ties
scored = scored.sort_values('risk_score', ascending=False).reset_index(drop=True)

scored.to_csv(outputs_dir / 'scored_members.csv', index=False)
print('Saved: {}  ({:,} rows x {} cols)'.format(
    outputs_dir / 'scored_members.csv', len(scored), len(scored.columns)))
print('Top 5 rows:')
print(scored[['member_id','risk_score','risk_score_calibrated','risk_tier','condition_cluster',
              'comorbidity_count','recommended_modality','nudge_signal']].head())

Saved: /home/alex/personal_projects/allied-health-nudge/outputs/scored_members.csv  (50,000 rows x 28 cols)
Top 5 rows:
                              member_id  risk_score  risk_score_calibrated  \
0  acc25029-406b-4afa-a15d-1c2baa775f6b    0.866012                    1.0   
1  763d43af-ad93-4885-8a45-b450ed3cb181    0.865782                    1.0   
2  1083ce86-367f-418e-97ec-aed919d08775    0.865697                    1.0   
3  6862cc07-b810-40f9-b1f2-fb723473b327    0.864461                    1.0   
4  d9629e6b-0f01-4f1e-9d0b-4b3c448e8aa4    0.864460                    1.0   

  risk_tier condition_cluster  comorbidity_count recommended_modality  \
0      High             Mixed                  3        Physiotherapy   
1      High             Mixed                  3           Psychology   
2      High             Mixed                  3           Psychology   
3      High             Mixed                  3           Psychology   
4      High             Mixed                 

## Step 7 — Scoring summary

In [7]:
summary = {
    'total_scored':         int(len(scored)),
    'high_risk':            int((scored['risk_tier'] == 'High').sum()),
    'medium_risk':          int((scored['risk_tier'] == 'Medium').sum()),
    'low_risk':             int((scored['risk_tier'] == 'Low').sum()),
    'benefit_gap':          int(scored['benefit_gap_flag'].sum()),
    'nudge_signal':         int(scored['nudge_signal'].sum()),
    'nudge_rate':           round(scored['nudge_signal'].mean(), 4),
    'plan_design_flag':     int(scored['plan_design_flag'].sum()),
    'threshold_high':       round(float(p90), 4),
    'threshold_medium':     round(float(p80), 4),
    'model_best_iteration': model.best_iteration,
    'roc_auc':              eval_metrics['roc_auc'],
    'pr_auc':               eval_metrics['pr_auc'],
    'precision_top20pct':   eval_metrics['precision_top20pct'],
    'recall_top20pct':      eval_metrics['recall_top20pct'],
}

with open(outputs_dir / 'scoring_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved: {}'.format(outputs_dir / 'scoring_summary.json'))

_sep = '─' * 52
print('\n' + _sep)
print('  SCORING SUMMARY')
print(_sep)
print('  {:<28}  {:>8,}'.format('Total scored', summary['total_scored']))
print('  {:<28}  {:>8,}  ({:.1f}%)'.format('High risk', summary['high_risk'],   summary['high_risk']/summary['total_scored']*100))
print('  {:<28}  {:>8,}  ({:.1f}%)'.format('Medium risk', summary['medium_risk'], summary['medium_risk']/summary['total_scored']*100))
print('  {:<28}  {:>8,}  ({:.1f}%)'.format('Low risk', summary['low_risk'],    summary['low_risk']/summary['total_scored']*100))
print(_sep)
print('  {:<28}  {:>8,}  ({:.1f}%)'.format('Nudge-eligible', summary['nudge_signal'], summary['nudge_rate']*100))
print('  {:<28}  {:>8,}'.format('Plan design flag', summary['plan_design_flag']))
print(_sep)

Saved: /home/alex/personal_projects/allied-health-nudge/outputs/scoring_summary.json

────────────────────────────────────────────────────
  SCORING SUMMARY
────────────────────────────────────────────────────
  Total scored                    50,000
  High risk                        5,000  (10.0%)
  Medium risk                      5,000  (10.0%)
  Low risk                        40,000  (80.0%)
────────────────────────────────────────────────────
  Nudge-eligible                   4,758  (9.5%)
  Plan design flag                     0
────────────────────────────────────────────────────


## Step 8 — Within-Cluster Tiering

The global tier assignment produces tiers dominated by a single cluster
(Mixed) because DGP base rates create a hierarchy the model faithfully
learns. To surface at-risk members from ALL clusters, we compute independent
p90/p80 thresholds per cluster.

This answers: "Who is the most at-risk MSK member?" rather than
"Who is the most at-risk member overall?"


In [8]:
# Per-cluster p90/p80 thresholds from raw scores
cluster_thresholds = {}
for cluster in scored['condition_cluster'].unique():
    mask = scored['condition_cluster'] == cluster
    cluster_scores = scored.loc[mask, 'risk_score']
    cluster_thresholds[cluster] = {
        'p90': np.percentile(cluster_scores, 90),
        'p80': np.percentile(cluster_scores, 80),
        'n': len(cluster_scores)
    }

def assign_cluster_tier(row):
    t = cluster_thresholds[row['condition_cluster']]
    if row['risk_score'] >= t['p90']:
        return 'High'
    elif row['risk_score'] >= t['p80']:
        return 'Medium'
    return 'Low'

scored['cluster_tier'] = scored.apply(assign_cluster_tier, axis=1)

print('Within-Cluster Tier Distribution:')
ct = pd.crosstab(scored['condition_cluster'], scored['cluster_tier'], margins=True)
print(ct)
print()
g = scored['risk_tier'].value_counts()
c = scored['cluster_tier'].value_counts()
print(f'Global:  High={g.get("High",0)}, Medium={g.get("Medium",0)}, Low={g.get("Low",0)}')
print(f'Cluster: High={c.get("High",0)}, Medium={c.get("Medium",0)}, Low={c.get("Low",0)}')


Within-Cluster Tier Distribution:
cluster_tier       High    Low  Medium    All
condition_cluster                            
Healthy             958   7664     958   9580
MH                  120    956     119   1195
MSK                 382   3054     382   3818
Metabolic          2136  17084    2136  21356
Mixed              1406  11240    1405  14051
All                5002  39998    5000  50000

Global:  High=5000, Medium=5000, Low=40000
Cluster: High=5002, Medium=5000, Low=39998


## Step 9 — Ranking Quality (NDCG@k)

NDCG@k penalises true positives appearing lower in the ranking.
Unlike recall/precision (binary: in top-k or not), NDCG cares about
ORDER within the top tier - the operational concern for a priority queue.


In [9]:
from sklearn.metrics import ndcg_score

labels = pd.read_parquet(data_dir / 'features' / 'labels.parquet')
swl = scored.merge(labels, on='member_id', how='inner')
y_true = swl['high_acute_risk'].values.reshape(1, -1)
y_score = swl['risk_score'].values.reshape(1, -1)

for pct in [5, 10, 20]:
    k = int(len(y_true[0]) * pct / 100)
    ndcg = ndcg_score(y_true, y_score, k=k)
    print(f'NDCG@{pct}% (k={k:,}): {ndcg:.4f}')


NDCG@5% (k=2,500): 0.5330
NDCG@10% (k=5,000): 0.4875
NDCG@20% (k=10,000): 0.4732


## Validation Checks

In [10]:
out = pd.read_csv(outputs_dir / 'scored_members.csv')

assert len(out) == 50_000,                              'Row count wrong'
assert out['member_id'].nunique() == 50_000,            'Duplicate member_ids'
assert out['risk_score'].between(0, 1).all(),           'risk_score out of [0,1]'
assert out['risk_tier'].isin(['High','Medium','Low']).all(), 'Unexpected tier values'
assert out['benefit_gap_flag'].isin([0,1]).all(),       'Non-binary benefit_gap_flag'
assert out['nudge_signal'].isin([0,1]).all(),           'Non-binary nudge_signal'
assert out['plan_design_flag'].isin([0,1]).all(),       'Non-binary plan_design_flag'
assert out['risk_score'].is_monotonic_decreasing,       'Not sorted by risk_score descending'

# High risk members should have higher average scores than Low
high_mean = out[out['risk_tier']=='High']['risk_score'].mean()
low_mean  = out[out['risk_tier']=='Low']['risk_score'].mean()
assert high_mean > low_mean, 'High risk mean score not greater than Low risk'

print('All Stage 5 validation checks passed')
print('  scored_members.csv: {:,} rows x {} cols'.format(len(out), len(out.columns)))
print('  High risk mean score: {:.4f}  |  Low risk mean score: {:.4f}'.format(high_mean, low_mean))

All Stage 5 validation checks passed
  scored_members.csv: 50,000 rows x 28 cols
  High risk mean score: 0.7688  |  Low risk mean score: 0.3439
